# AI 103: Retrieval-Augmented Generation (RAG)

## Learning objectives

By the end of this notebook, students will be able to:

- explain the difference between a language model's pretrained knowledge and information retrieved from local documents;
- identify the stages of a basic RAG pipeline;
- extract text from PDF files and divide it into overlapping chunks;
- create numerical document embeddings and store them in a vector index;
- retrieve chunks that are relevant to a question;
- construct a prompt containing both retrieved context and the user's question; and
- compare an LLM-only answer with a RAG-supported answer.

## The question we are investigating

An LLM answers a question from the patterns learned during training. That knowledge may be incomplete, outdated, or missing the details in documents supplied for this exercise. RAG addresses this limitation by **retrieving relevant source text at question time** and adding that text to the prompt.

In this lab, an answer is therefore generated in two distinct modes:

1. **LLM only:** the question is sent directly to the model.
2. **RAG:** relevant PDF text is retrieved first, then sent to the model with the question.

> RAG does not automatically guarantee correctness. The quality of retrieval, chunking, embeddings, prompting, and the base model all affect the final answer.

## 1. Environment setup

The following libraries provide the tools used throughout the notebook:

- `faiss-cpu` builds and searches a vector index.
- `sentence-transformers` converts text into embeddings.
- `transformers` loads the tokenizer and text-generation model.
- `PyPDF2` reads text from PDF files.
- `ipywidgets` creates the interactive question interface later in the notebook.

This cell only needs to be run once per Python environment. In Jupyter, a leading `!` sends the command to the environment's shell.

In [ ]:
# Install the Python packages required by this notebook.
# Re-run this cell after recreating or changing the notebook environment.
%pip install faiss-cpu sentence-transformers transformers PyPDF2 ipywidgets

## 2. Choose local storage locations

Model files are cached in `MODELS_DIR`, and uploaded PDFs are stored in `UPLOADS_DIR`. Using the user's home directory keeps these files persistent between notebook sessions. The directories are created if they do not already exist.

Do not place secrets or sensitive documents in a public or shared upload directory.

In [ ]:
import os
from pathlib import Path

HOME = Path.home()

# Cache downloaded model weights here.
MODELS_DIR = HOME / "ai_models"

# Place source PDFs here; the notebook will process files from this folder.
UPLOADS_DIR = HOME / "ai_uploads"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
UPLOADS_DIR.mkdir(parents=True, exist_ok=True)

print("Model cache:", MODELS_DIR)
print("PDF upload folder:", UPLOADS_DIR)

## 3. Upload source documents

### JupyterLab instructions

1. Open the **File Browser** from the left sidebar.
2. Navigate to `UPLOADS_DIR`, shown by the previous cell.
3. Upload several PDF documents.

The examples below span networking, security, cloud, operating systems, and RAG concepts. For a useful comparison, include at least one PDF that clearly contains answers to the questions in this notebook.

### Suggested source documents

- NIST Cybersecurity Framework: <https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf>
- NIST SP 800-53: <https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-53r5.pdf>
- OWASP Top 10: <https://owasp.org/www-project-top-ten/>
- Cisco IOS XE OSPF configuration guide
- Cisco IOS XE BGP configuration guide
- Cisco VLAN, NAT, and routing guides
- Azure Virtual Networks, VPN Gateway, and Load Balancer documentation
- Linux `systemd`, SSH, `firewalld`, and SELinux documentation
- RAG survey paper: <https://arxiv.org/pdf/2312.10997>
- Embeddings or vector database articles

The upload step is intentionally separate from the code. This makes it clear that RAG uses a local **knowledge source**, not the LLM's pretrained knowledge.

## 4. Detect available compute

PyTorch will use a CUDA-enabled GPU when one is available. Otherwise, it will use the CPU. GPU acceleration is desirable for model inference and embedding creation, but the notebook may still work on a CPU with a smaller model.

The `DEVICE` variable is reused by both the text-generation model and the embedding model.

In [ ]:
# Check whether PyTorch can use an NVIDIA CUDA GPU.
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())
print("Selected device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1),
        "GB",
    )

## 5. Load the base language model

The current default is `google/flan-t5-base`, an instruction-tuned sequence-to-sequence model.

- `AutoTokenizer` converts text into token IDs and prepares the model input.
- `AutoModelForSeq2SeqLM` loads the model designed to generate a text sequence from an input sequence.
- `cache_dir` stores downloaded weights in `MODELS_DIR`.
- `.to(DEVICE)` moves the model to CPU or GPU.

Only one model should be uncommented at a time.

The commented **Intermediate** and **Advanced** alternatives such as BGE and Qwen are not drop-in replacements for this sequence-to-sequence model. BGE models are embedding models, while Qwen models are generative models that require a different loader and, in some environments, more memory. They can be explored in later extensions of this lab.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Beginner option: good balance for a classroom lab.
model_name = "google/flan-t5-base"

# Smaller CPU fallback:
# model_name = "google/flan-t5-small"

# Alternative sequence-to-sequence model:
# model_name = "facebook/bart-base"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=str(MODELS_DIR),
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    cache_dir=str(MODELS_DIR),
).to(DEVICE)

model.eval()

print("Loaded model:", model_name)
print("Model device:", next(model.parameters()).device)
print("Parameter count:", f"{model.num_parameters():,}")

## 6. Establish a baseline: answer questions without RAG

First, the notebook sends each question directly to the model. This establishes a **baseline**: what the model can answer using only its pretrained parameters and the text provided in the question.

No document context is included in this stage. The resulting answers may:

- be accurate but lack document-specific detail;
- be plausible but unsupported;
- reference the wrong standard or product;
- hallucinate details; or
- fail to answer because the topic was not learned reliably.

The baseline makes the RAG comparison fair and visible.

In [ ]:
questions = [
    "How do I enable OSPF on an interface?",
    "What is the difference between OSPF area 0 and a stub area?",
    "What Cisco commands verify BGP neighbors?",
    "What is the principle of least privilege?",
    "Which CIS control addresses vulnerability management?",
    "What are the major OWASP web application risks?",
    "What is the difference between a Load Balancer and an Application Gateway?",
    "How does zone redundancy work?",
    "What are the components of a virtual network?",
    "How do I restart a service?",
    "How does SELinux differ from standard Unix permissions?",
    "What is the purpose of systemd?",
    "What are embeddings?",
    "How does retrieval augmentation work?",
    "What is the purpose of chunking?",
    "Why does overlap improve retrieval?",
]

In [ ]:
def ask(model, question):
    """Generate an answer using only the question and the base model.

    Args:
        model: Loaded text-generation model.
        question: The user's question as plain text.

    Returns:
        The decoded answer string.
    """
    inputs = tokenizer(question, return_tensors="pt").to(DEVICE)

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        num_beams=4,
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("Q:", question)
    print("A:", answer)
    print("-" * 80)
    return answer

### Run the baseline

`%%time` reports the elapsed wall-clock time for this entire loop. This is useful for comparing the cost of model-only responses with RAG responses, but actual runtime varies by hardware, model size, PDF count, and settings.

In [ ]:
%%time

for question in questions:
    ask(model, question)

## 7. Extract text from uploaded PDFs

PDF files can contain:

- selectable text;
- scanned images without embedded text;
- multiple columns;
- tables;
- headers and footers; or
- unusual characters.

PyPDF2 iterates through each page and calls `extract_text()`. Scanned PDFs will require an OCR step before text extraction can work, and extraction quality still depends on how the PDF was created.

The notebook reports every PDF it finds and stops with an explicit message if no usable PDF is available.

In [ ]:
import PyPDF2

documents = []

pdf_files = sorted(
    path for path in UPLOADS_DIR.iterdir()
    if path.is_file() and path.suffix.lower() == ".pdf"
)

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files were found in {UPLOADS_DIR}. Upload at least one PDF and rerun this cell."
    )

for filepath in pdf_files:
    text_by_page = []

    with filepath.open("rb") as pdf_handle:
        reader = PyPDF2.PdfReader(pdf_handle)

        for page_number, page in enumerate(reader.pages, start=1):
            page_text = page.extract_text()
            if page_text:
                text_by_page.append(page_text)

    document_text = "\n".join(text_by_page)

    if document_text.strip():
        documents.append((filepath.name, document_text))
        print(f"Loaded: {filepath.name} ({len(reader.pages)} pages)")
    else:
        print(
            f"Skipped: {filepath.name} - no selectable text was found "
            "(OCR may be required)."
        )

if not documents:
    raise ValueError(
        "None of the uploaded PDFs contained selectable text. "
        "OCR must be performed before extraction."
    )

print(f"\nDocuments with usable text: {len(documents)}")

## 8. Split documents into chunks

An embedding model converts an entire chunk into a fixed-length vector. Chunks are units of text that can be retrieved and passed to the language model.

Chunk size involves a tradeoff:

| Chunk size | Effect |
|---|---|
| 100-300 characters | May split ideas into fragments but provide precise matches |
| 500-1000 characters | Good starting point for introductory RAG labs |
| 1000-2000 characters | Often preserves more context for a product or policy document |
| 2000-4000 characters | May include more complete ideas but reduce retrieval precision |

### Overlap

Overlap repeats a portion of neighboring chunks. It helps preserve an idea that might be split at a boundary. For example, if `chunk_size = 500` and `overlap = 200`, the next chunk begins 300 characters after the previous chunk begins.

In [ ]:
chunks = []

chunk_size = 500
overlap = 200

if overlap >= chunk_size:
    raise ValueError("overlap must be smaller than chunk_size")

step = chunk_size - overlap

for filename, text in documents:
    for start in range(0, len(text), step):
        chunk_text = text[start:start + chunk_size].strip()

        if len(chunk_text) < 100:
            continue

        chunks.append({
            "source": filename,
            "start": start,
            "end": start + len(chunk_text),
            "text": chunk_text,
        })

if not chunks:
    raise ValueError("No chunks met the minimum length. Check the extracted PDF text.")

print("Created chunks:", len(chunks))
print("Chunk size:", chunk_size)
print("Overlap:", overlap)
print("Step between chunk starts:", step)

print("\nFirst chunk preview:")
print("Source:", chunks[0]["source"])
print(chunks[0]["text"][:300])

## 9. Choose and load an embedding model

An embedding model maps text to a vector, such as a list of floating-point numbers. Similar meanings should usually produce vectors that are close together in vector space.

The notebook uses `all-MiniLM-L6-v2` by default. It is relatively small and suitable for CPU or GPU experiments.

Embedding models do not generate answers. They are used only to represent the meaning of text for retrieval.

The alternatives are examples of what a later experiment might use. Load one embedding model only.

In [ ]:
from sentence_transformers import SentenceTransformer

# Baseline embedding model: small and relatively fast.
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    cache_folder=str(MODELS_DIR),
    device=DEVICE,
)

# Slightly higher-quality alternatives:
# embedder = SentenceTransformer(
#     "sentence-transformers/all-mpnet-base-v2",
#     cache_folder=str(MODELS_DIR),
#     device=DEVICE,
# )
# embedder = SentenceTransformer(
#     "intfloat/e5-small-v2",
#     cache_folder=str(MODELS_DIR),
#     device=DEVICE,
# )
# embedder = SentenceTransformer(
#     "BAAI/bge-small-en-v1.5",
#     cache_folder=str(MODELS_DIR),
#     device=DEVICE,
# )

print("Embedding model:", embedder.get_sentence_embedding_dimension(), "dimensions")

# convert_to_numpy makes the result directly usable by FAISS.
chunk_embeddings = embedder.encode(
    [chunk["text"] for chunk in chunks],
    convert_to_numpy=True,
    show_progress_bar=True,
)

print("Embedding matrix shape:", chunk_embeddings.shape)
print("This means:")
print("-", len(chunks), "chunks")
print("-", chunk_embeddings.shape[1], "numbers per chunk")

## 10. Build the FAISS vector index

The embedding matrix has one row for each chunk and one column for each embedding dimension.

FAISS stores all chunk vectors and performs a similarity search. `IndexFlatL2` measures Euclidean (L2) distance:

- a distance of `0` means identical vectors;
- smaller distances indicate more similar vectors; and
- larger distances indicate less similar vectors.

This is a simple in-memory index. It must be rebuilt when the notebook restarts or the uploaded PDFs change.

In [ ]:
import faiss
import numpy as np

dimension = chunk_embeddings.shape[1]

if chunk_embeddings.shape[0] != len(chunks):
    raise ValueError(
        "The number of embedding rows does not match the number of chunks."
    )

# IndexFlatL2 performs an exact L2-distance search.
index = faiss.IndexFlatL2(dimension)

embedding_array = np.asarray(chunk_embeddings, dtype="float32")
index.add(embedding_array)

print("FAISS index vectors:", index.ntotal)
print("Embedding dimension:", dimension)

## 11. Retrieve relevant chunks

Retrieval has the following steps:

1. Convert the question to an embedding with the same model used for documents.
2. Search the FAISS index for the `k` closest vectors.
3. Return the matching chunk text, source document, character position, and distance.

`k` controls how much context is retrieved. Larger values include more text but can introduce unrelated material and consume the model's context window. A classroom starting range is 3-5.

In [ ]:
def retrieve(question, k=5):
    """Retrieve the k chunks nearest to a question.

    Args:
        question: Plain-text question to embed and search.
        k: Number of chunks to retrieve.

    Returns:
        A list of dictionaries containing chunk metadata and L2 distance.
    """
    if k < 1:
        raise ValueError("k must be at least 1.")
    if k > len(chunks):
        k = len(chunks)

    question_embedding = embedder.encode(
        [question],
        convert_to_numpy=True,
    )

    distances, indices = index.search(
        np.asarray(question_embedding, dtype="float32"),
        k,
    )

    results = []
    for position, chunk_index in enumerate(indices[0]):
        if chunk_index == -1:
            continue

        result = dict(chunks[int(chunk_index)])
        result["distance"] = float(distances[0][position])
        results.append(result)

    return results

## 12. Generate an answer with RAG

The RAG function combines retrieval with generation:

1. Retrieve the `k` closest chunks.
2. Join their text into a context block.
3. Build an instruction that asks the model to use that context.
4. Tokenize and submit the full prompt.
5. Generate and decode the answer.
6. Print the source PDFs that contributed retrieved chunks.

Because FAISS retrieves the closest vectors, it does not prove that a chunk contains the correct answer. The prompt directs the model to ground its response, but the model can still make mistakes.

Set `SHOW_RETRIEVED_CHUNKS=True` to show exactly what the retriever supplied.

In [ ]:
def ask_rag(model, question, k=3, SHOW_RETRIEVED_CHUNKS=False):
    """Retrieve local document context and generate a grounded answer.

    Args:
        model: Loaded text-generation model.
        question: The user's question.
        k: Number of document chunks to retrieve.
        SHOW_RETRIEVED_CHUNKS: Whether to print retrieved text and metadata.

    Returns:
        A dictionary containing the answer and retrieved chunks.
    """
    retrieved = retrieve(question, k=k)
    context = "\n\n".join(chunk["text"] for chunk in retrieved)

    if SHOW_RETRIEVED_CHUNKS:
        print("\nRetrieved chunks")
        print("-" * 50)

        for i, chunk in enumerate(retrieved, start=1):
            print(f"[Chunk {i}]")
            print(f"Source: {chunk['source']}")
            print(
                f"Characters: {chunk['start']}-{chunk['end']}; "
                f"L2 distance: {chunk['distance']:.4f}"
            )
            print(chunk["text"][:500])
            print()

    prompt = (
        "Use only the context below to answer the question. "
        "If the context does not contain the answer, say that it is not provided "
        "instead of guessing.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        num_beams=4,
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    sources = sorted({chunk["source"] for chunk in retrieved})

    print("Q:", question)
    print("A:", answer)
    print("\nSources:")
    if sources:
        for source in sources:
            print("-", source)
    else:
        print("- No chunks retrieved")
    print("-" * 80, "\n")

    return {
        "answer": answer,
        "sources": sources,
        "retrieved": retrieved,
        "prompt": prompt,
    }

## 13. Run the RAG baseline

Each saved question is answered again, this time with the three closest document chunks included in the prompt. Compare these outputs with the LLM-only section above.

For each question, consider:

- Did the retrieved source contain relevant information?
- Did the model use the source or ignore it?
- Did RAG improve factual accuracy?
- Did RAG introduce a new error?
- Did retrieval find the right document but the wrong chunk?
- Did a more general chunk compete with a more precise chunk?

In [ ]:
%%time

for question in questions:
    ask_rag(model, question, k=3, SHOW_RETRIEVED_CHUNKS=False)

## 14. Compare LLM-only and RAG interactively

The widget below accepts a custom question and runs it in both modes:

- **LLM only** calls `ask()` without document context.
- **RAG** calls `ask_rag()` and prints the retrieved chunks.

The comparison also measures generation and retrieval runtime for the individual question. Runtime is shown for interpretation, not as a guarantee of model quality.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import time

question_box = widgets.Text(
    placeholder="Ask a question...",
    description="Question:",
    layout=widgets.Layout(width="80%"),
)

button = widgets.Button(
    description="Compare answers",
    button_style="primary",
)

output = widgets.Output()


def submit_question(_):
    question = question_box.value.strip()
    if not question:
        print("Enter a question before selecting Compare answers.")
        return

    with output:
        output.clear_output(wait=True)

        print("=" * 70)
        print("QUESTION")
        print("=" * 70)
        print(question)

        print("\n" + "=" * 70)
        print("LLM ONLY")
        print("=" * 70)

        llm_start = time.time()
        try:
            ask(model, question)
        except Exception as error:
            print(f"LLM-only error: {type(error).__name__}: {error}")
        llm_time = time.time() - llm_start
        print(f"Runtime: {llm_time:.2f} sec")

        print("\n" + "=" * 70)
        print("RAG")
        print("=" * 70)

        rag_start = time.time()
        try:
            ask_rag(
                model,
                question,
                k=3,
                SHOW_RETRIEVED_CHUNKS=True,
            )
        except Exception as error:
            print(f"RAG error: {type(error).__name__}: {error}")
        rag_time = time.time() - rag_start
        print(f"Runtime: {rag_time:.2f} sec")

        print("\n" + "=" * 70)
        print("COMPARISON")
        print("=" * 70)
        print(f"LLM only: {llm_time:.2f} sec")
        print(f"RAG:      {rag_time:.2f} sec")
        print(f"Difference: {rag_time - llm_time:.2f} sec")


button.on_click(submit_question)

display(question_box)
display(button)
display(output)

## 15. Cleanup and next experiments

Cleanup is optional and intentionally disabled. Removing `MODELS_DIR` deletes cached model weights and may require downloading them again. Removing `UPLOADS_DIR` deletes source documents.

Before rerunning from the beginning, rerun the upload/extraction/chunking/embedding section whenever source documents change.

### Classroom experiments

Try changing one variable at a time and record observations:

1. **Chunk size:** use 300, 500, and 1000 characters.
2. **Overlap:** compare 0, 100, and 200 characters.
3. **Retrieved count:** test `k=1`, `k=3`, and `k=5`.
4. **Embedding model:** compare MiniLM with another embedding model.
5. **Prompt:** add or remove the instruction to avoid guessing.
6. **Source set:** upload a small document that clearly answers one question.
7. **Baseline:** remove a source PDF and ask about it again.

This separation lets students distinguish failures in retrieval from failures in generation.

In [ ]:
# The cleanup commands are intentionally commented out.
# Uncomment and run only when temporary files are no longer needed.

# import shutil

# shutil.rmtree(MODELS_DIR)
# shutil.rmtree(UPLOADS_DIR)

# print("Temporary models and uploads removed")

# Notebook summary

The full RAG workflow is:

```text
PDF files
   |
   v
Text extraction
   |
   v
Overlapping text chunks
   |
   v
Document embeddings
   |
   v
FAISS vector index
   |
   v
Question embedding --> nearest chunks --> context prompt --> LLM answer
```

The baseline and RAG modes differ in one important way: the baseline sends only the question, while RAG sends a prompt containing both retrieved document text and the question.